# 🛒 Vendas em E-commerce no Brasil — Projeto G1

## 1. Introdução e contextualização

O comércio eletrônico tornou-se um importante canal de vendas no Brasil e produz dados sobre vendas, produtos, regiões, faturamento, logística e comportamento de consumo.

Este projeto investiga vendas de e-commerce entre **2015 e 2024**, buscando responder:

- Quais categorias possuem maior faturamento?
- Quais estados concentram mais vendas?
- Existem períodos sazonais de maior consumo?
- Quais canais possuem melhor desempenho?
- Quais produtos apresentam maior volume de vendas?
- Como o faturamento evoluiu?
- Existem regiões com maior ticket médio?


## 2. Explicação da base

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns",None)

df=pd.read_csv("../dados/simulacao_ecommerce_brasil.csv")
df["data"]=pd.to_datetime(df["data"],errors="coerce")
df.head()


In [ ]:
print("Dimensões:",df.shape)
print("Período:",df.data.min().date(),"a",df.data.max().date())
print("\nColunas:")
print(df.columns.tolist())
print("\nTipos:")
print(df.dtypes)


## 3. Limpeza e preparação

In [ ]:
print("Valores ausentes:")
display(df.isna().sum())
print("Duplicidades:",df.duplicated().sum())
print("Datas inválidas:",df.data.isna().sum())

for c in ["regiao","uf","cidade","canal_venda","categoria","produto"]:
    df[c]=df[c].astype("string").str.strip()


A base foi verificada quanto a ausências, duplicidades e datas. Os campos de texto foram padronizados e a data convertida para `datetime`.

## 4. Engenharia de atributos

In [ ]:
df["ano"]=df.data.dt.year
df["mes"]=df.data.dt.month
df["trimestre"]=df.data.dt.quarter
df["ano_mes"]=df.data.dt.to_period("M").astype(str)
df["margem_lucro"]=np.where(df.faturamento!=0,df.lucro/df.faturamento*100,0)

# Ticket médio operacional: faturamento dividido pela quantidade de unidades vendidas.
df["ticket_medio"]=np.where(df.quantidade!=0,df.faturamento/df.quantidade,0)
display(df.head())


## 5. KPIs

In [ ]:
faturamento=df.faturamento.sum()
lucro=df.lucro.sum()
quantidade=df.quantidade.sum()

kpis=pd.Series({
    "Faturamento total":faturamento,
    "Lucro total":lucro,
    "Ticket médio":faturamento/quantidade,
    "Margem de lucro (%)":lucro/faturamento*100,
    "Avaliação média":df.avaliacao_cliente.mean(),
    "Prazo médio (dias)":df.prazo_entrega.mean()
})
display(kpis)
print("Produto mais vendido:",df.groupby("produto").quantidade.sum().idxmax())
print("Categoria mais lucrativa:",df.groupby("categoria").lucro.sum().idxmax())
print("Região com maior faturamento:",df.groupby("regiao").faturamento.sum().idxmax())


## 6. Evolução temporal

In [ ]:
mensal=df.groupby("data",as_index=False).agg(faturamento=("faturamento","sum"),lucro=("lucro","sum"))
plt.figure(figsize=(13,5))
plt.plot(mensal.data,mensal.faturamento,label="Faturamento")
plt.plot(mensal.data,mensal.lucro,label="Lucro")
plt.title("Evolução do faturamento e lucro")
plt.xlabel("Data");plt.ylabel("R$");plt.legend()
plt.xticks(rotation=30);plt.tight_layout();plt.show()


## 7. Comparação regional

In [ ]:
estado=df.groupby("uf",as_index=False).agg(faturamento=("faturamento","sum"),lucro=("lucro","sum"),quantidade=("quantidade","sum")).sort_values("faturamento",ascending=False)
plt.figure(figsize=(12,7))
sns.barplot(data=estado,x="faturamento",y="uf")
plt.title("Faturamento por estado")
plt.xlabel("R$");plt.ylabel("UF");plt.tight_layout();plt.show()


In [ ]:
reg=df.groupby("regiao",as_index=False).agg(faturamento=("faturamento","sum"),quantidade=("quantidade","sum"))
reg["ticket_medio"]=reg.faturamento/reg.quantidade
display(reg.sort_values("ticket_medio",ascending=False))


## 8. Categorias, produtos e canais

In [ ]:
cat=df.groupby("categoria",as_index=False).agg(faturamento=("faturamento","sum"),lucro=("lucro","sum"),quantidade=("quantidade","sum")).sort_values("faturamento",ascending=False)
fig,ax=plt.subplots(figsize=(10,5))
sns.barplot(data=cat,x="faturamento",y="categoria",ax=ax)
ax.set_title("Faturamento por categoria");ax.set_xlabel("R$");ax.set_ylabel("")
plt.tight_layout();plt.show()


In [ ]:
prod=df.groupby("produto",as_index=False).agg(quantidade=("quantidade","sum"),faturamento=("faturamento","sum"),lucro=("lucro","sum")).sort_values("quantidade",ascending=False)
display(prod)
print("Produto mais vendido:",prod.iloc[0]["produto"])
print("Categoria mais lucrativa:",cat.sort_values("lucro",ascending=False).iloc[0]["categoria"])


In [ ]:
canal=df.groupby("canal_venda",as_index=False).agg(faturamento=("faturamento","sum"),lucro=("lucro","sum")).sort_values("faturamento",ascending=False)
display(canal)


## 9. Análise financeira — lucro × faturamento

In [ ]:
lf=df.groupby(["uf","regiao"],as_index=False).agg(faturamento=("faturamento","sum"),lucro=("lucro","sum"))
plt.figure(figsize=(10,6))
sns.scatterplot(data=lf,x="faturamento",y="lucro",hue="regiao",s=100)
plt.title("Relação entre faturamento e lucro por estado")
plt.xlabel("Faturamento (R$)");plt.ylabel("Lucro (R$)")
plt.tight_layout();plt.show()


## 10. Análise logística e avaliação

In [ ]:
log=df.groupby("uf",as_index=False).agg(
    prazo_medio=("prazo_entrega","mean"),
    avaliacao=("avaliacao_cliente","mean"),
    quantidade=("quantidade","sum")
)
display(log.sort_values("prazo_medio"))


In [ ]:
plt.figure(figsize=(10,5))
sns.scatterplot(data=log,x="quantidade",y="avaliacao",size="prazo_medio",hue="prazo_medio",sizes=(60,300))
plt.title("Quantidade vendida × avaliação média")
plt.xlabel("Quantidade vendida");plt.ylabel("Avaliação")
plt.tight_layout();plt.show()


## 11. Sazonalidade — heatmap mensal

In [ ]:
heat=df.pivot_table(index="ano",columns="mes",values="faturamento",aggfunc="sum",fill_value=0)
plt.figure(figsize=(13,6))
sns.heatmap(heat,annot=True,fmt=".0f",cmap="YlGnBu")
plt.title("Heatmap de faturamento por ano e mês")
plt.xlabel("Mês");plt.ylabel("Ano")
plt.tight_layout();plt.show()


O heatmap permite identificar meses e anos com maior concentração de faturamento e investigar padrões sazonais.

## 12. Interpretação dos resultados

In [ ]:
print("Maior categoria por faturamento:",df.groupby("categoria").faturamento.sum().idxmax())
print("Maior categoria por lucro:",df.groupby("categoria").lucro.sum().idxmax())
print("Produto mais vendido:",df.groupby("produto").quantidade.sum().idxmax())
print("Estado líder:",df.groupby("uf").faturamento.sum().idxmax())
print("Região líder:",df.groupby("regiao").faturamento.sum().idxmax())
print("Canal líder:",df.groupby("canal_venda").faturamento.sum().idxmax())


### Interpretação

Os resultados devem ser lidos em conjunto. A categoria com maior faturamento não necessariamente é a de maior lucro. Da mesma forma, o estado com maior faturamento pode não apresentar o maior ticket médio.

A linha temporal mostra a evolução das vendas. O heatmap evidencia a sazonalidade mensal. O ranking de produtos identifica o maior volume vendido. A análise regional mostra concentração geográfica e permite comparar ticket médio. A dispersão lucro × faturamento ajuda a avaliar eficiência financeira. A análise logística relaciona prazo de entrega e avaliação do cliente.


## 13. Conclusão

O projeto transforma a base de vendas em uma aplicação analítica. Foram realizados tratamento, engenharia de atributos, KPIs, análises temporal, regional, de categorias, produtos, canais, logística, avaliação e sazonalidade.

O dashboard Streamlit disponibiliza filtros por ano, mês, região, estado, categoria e canal, permitindo explorar os resultados de forma interativa. A aplicação também oferece tabela dinâmica e download dos dados filtrados.

A solução atende às perguntas orientadoras e aos requisitos específicos de visualização e dashboard do projeto.
